# Cedar Crumb
**Good bread. Time well spent.**

Upload **C5_Cedar_Crumb_Submission.zip** to **/content**, enable **GEMINI_API_KEY** in Colab Secrets, then select **Runtime → Run all**.

Runtime: CPU · Python 3.12

Chat requests use Gemini API quota. The temporary application link is accessible to anyone with the link while the runtime is running.


## 1. Environment


In [ ]:
from pathlib import Path
import os, subprocess, sys, zipfile
from uuid import uuid4

archive_path = Path('/content/C5_Cedar_Crumb_Submission.zip')
if not archive_path.is_file():
    from google.colab import files
    print('Select C5_Cedar_Crumb_Submission.zip')
    os.chdir('/content')
    files.upload()
if not archive_path.is_file():
    raise FileNotFoundError('Upload C5_Cedar_Crumb_Submission.zip into /content.')
run_root = Path('/content') / f'cedar-run-{uuid4().hex[:8]}'
run_root.mkdir()
with zipfile.ZipFile(archive_path) as archive:
    for entry in archive.infolist():
        if not (run_root / entry.filename).resolve().is_relative_to(run_root.resolve()):
            raise ValueError('Unsafe archive path.')
    archive.extractall(run_root)
PROJECT = run_root / 'cedar_crumb_c5'
assert (PROJECT / 'app.py').is_file(), 'Upload the current submission ZIP containing app.py.'

def command(args):
    with subprocess.Popen(args, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        status = process.wait()
    if status:
        raise RuntimeError(f'Command failed (exit {status}); see output above.')

command([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv==0.13.6'])
UV = [sys.executable, '-m', 'uv']
ENV = Path('/content/cedar_crumb_py312')
command(UV + ['venv', '--python', '3.12', '--managed-python', '--allow-existing', str(ENV)])
PYTHON = str(ENV / 'bin/python')
command(UV + ['pip', 'install', '--python', PYTHON, '-r', 'requirements-dev.txt'])
command(UV + ['pip', 'check', '--python', PYTHON])
if not os.environ.get('GEMINI_API_KEY'):
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
assert os.environ.get('GEMINI_API_KEY'), 'Enable notebook access for GEMINI_API_KEY in Colab Secrets.'
os.environ.setdefault('GEMINI_MODEL', 'gemini-3.5-flash-lite')
print('Cedar Crumb installed. Key loaded privately.')


## 2. Input and output contracts · models.py


In [ ]:
%%writefile "{PROJECT}/models.py"
"""Input contracts and structured results for Cedar Crumb's MCP tools."""
from typing import Annotated, Literal

from pydantic import BaseModel, ConfigDict, Field

Query = Annotated[str, Field(max_length=120, description="Case-insensitive product, flavor, or ingredient substring. Use an empty string for the full menu.")]
ProductID = Annotated[str, Field(min_length=1, max_length=60, description="Exact product ID returned by search_menu, such as plain or olive-rosemary.")]
Quantity = Annotated[int, Field(strict=True, ge=1, le=30, description="Whole loaves, 1-30. Larger requests require a staff quote.")]
RequestID = Annotated[str, Field(pattern=r"^[A-Za-z0-9_-]{8,64}$", description="Unique request reference; reuse this reference only when retrying identical details.")]
PickupAt = Annotated[str, Field(description="Pickup date/time in ISO 8601 with a UTC offset, for example 2026-11-04T10:00:00+02:00. At least 24 hours ahead, Monday-Saturday, 07:00 to before 19:00 Beirut time.")]


class Customer(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)
    name: str = Field(min_length=1, max_length=80, description="Customer-provided name.")
    email: str = Field(max_length=254, pattern=r"^[^\s@]+@[^\s@]+\.[^\s@]+$", description="Customer-provided email for pickup follow-up.")
    consent: bool = Field(strict=True, description="True only after the customer agrees to save these details for pickup follow-up.")


class Product(BaseModel):
    id: str
    name: str
    weight_g: int
    price_usd: str
    description: str
    ingredients: str
    allergens: list[str]


class MenuResult(BaseModel):
    query: str
    products: list[Product]
    currency: Literal["USD"] = "USD"
    hours: str
    pickup_policy: str
    allergen_notice: str


class PickupInquiry(BaseModel):
    inquiry_id: str
    request_id: str
    product_id: str
    product_name: str
    quantity: int
    unit_price_usd: str
    estimated_total_usd: str
    currency: Literal["USD"] = "USD"
    customer_name: str
    pickup_at: str
    status: Literal["pending_confirmation"] = "pending_confirmation"
    already_saved: bool = False


class PreparedInquiry(BaseModel):
    query: str
    steps: list[str]
    inquiry: PickupInquiry


## 3. Menu and inquiry operations · bakery.py


In [ ]:
%%writefile "{PROJECT}/bakery.py"
"""Menu retrieval and durable pickup inquiry operations."""
from __future__ import annotations

import hashlib
import json
import re
import sqlite3
from contextlib import closing
from datetime import UTC, datetime, time, timedelta
from decimal import Decimal
from pathlib import Path
from typing import Callable
from uuid import uuid4
from zoneinfo import ZoneInfo

from models import Customer, MenuResult, PickupInquiry, PreparedInquiry, Product

ROOT = Path(__file__).resolve().parent
BEIRUT = ZoneInfo("Asia/Beirut")


class BakeryError(Exception):
    def __init__(self, code: str, message: str):
        self.code = code
        super().__init__(f"{code}: {message}")


class Bakery:
    def __init__(self, database: Path | str, *, catalog: Path | str = ROOT / "data" / "bakery.json", now: Callable[[], datetime] | None = None):
        self.database = Path(database).resolve()
        self.catalog = Path(catalog)
        self.now = now or (lambda: datetime.now(UTC))
        self.data, self.products = self._read_catalog()

    def _read_catalog(self):
        try:
            data = json.loads(self.catalog.read_text(encoding="utf-8"))
            products = [Product(**{**p, "price_usd": f"{Decimal(str(p['price_usd'])):.2f}"}) for p in data["products"]]
            return data, products
        except (OSError, ValueError, KeyError, TypeError) as exc:
            raise BakeryError("CATALOG_UNAVAILABLE", "The bakery menu could not be loaded.") from exc

    def initialize(self) -> None:
        """Prepare storage at server startup, outside the Data operation."""
        try:
            self.database.parent.mkdir(parents=True, exist_ok=True)
            with closing(self._connect()) as connection, connection:
                connection.execute("""CREATE TABLE IF NOT EXISTS pickup_inquiries (
                    request_id TEXT PRIMARY KEY,
                    fingerprint TEXT NOT NULL,
                    customer_email TEXT NOT NULL,
                    result_json TEXT NOT NULL)""")
        except (OSError, sqlite3.Error) as exc:
            raise BakeryError("STORAGE_UNAVAILABLE", "Pickup inquiry storage could not be initialized.") from exc

    def _connect(self, readonly: bool = False) -> sqlite3.Connection:
        target = self.database.as_uri() + "?mode=ro" if readonly else str(self.database)
        connection = sqlite3.connect(target, uri=readonly, timeout=5)
        connection.row_factory = sqlite3.Row
        return connection

    def search_menu(self, query: str) -> MenuResult:
        """Retrieve documented menu facts without touching inquiry storage."""
        if not isinstance(query, str) or len(query) > 120 or (query and not query.strip()):
            raise BakeryError("INVALID_QUERY", "Use a product or ingredient query of up to 120 characters, or an empty string for the full menu.")
        query = query.strip()
        data, catalog_products = self._read_catalog()
        products = [p for p in catalog_products if not query or query.casefold() in " ".join(
            [p.id, p.name, p.description, p.ingredients, *p.allergens]
        ).casefold()]
        policies = data["policies"]
        return MenuResult(query=query, products=products, hours=policies["hours"], pickup_policy=policies["ordering"], allergen_notice=policies["allergens"])

    def _product(self, product_id: str) -> Product:
        _, products = self._read_catalog()
        for product in products:
            if product.id == product_id:
                return product
        raise BakeryError("PRODUCT_NOT_FOUND", "Unknown product ID. Look up the menu before saving a request.")

    @staticmethod
    def _pickup(pickup_at: str) -> datetime:
        try:
            result = datetime.fromisoformat(pickup_at)
        except (ValueError, TypeError) as exc:
            raise BakeryError("INVALID_PICKUP", "Use an ISO 8601 pickup date/time with a UTC offset.") from exc
        if result.tzinfo is None or result.utcoffset() is None:
            raise BakeryError("INVALID_PICKUP", "Include the pickup date/time's UTC offset.")
        return result.astimezone(BEIRUT)

    def save_pickup_inquiry(self, product_id: str, quantity: int, customer: Customer, pickup_at: str, request_id: str) -> PickupInquiry:
        """Persist one validated pickup inquiry; identical retries return its ID."""
        product = self._product(product_id)
        if type(quantity) is not int or not 1 <= quantity <= 30:
            raise BakeryError("INVALID_QUANTITY", "Request 1-30 whole loaves; larger quantities require a staff quote.")
        if not isinstance(customer, Customer):
            raise BakeryError("INVALID_CUSTOMER", "Supply a customer name, email, and consent.")
        if not customer.consent:
            raise BakeryError("CONSENT_REQUIRED", "Customer consent is required before saving contact details.")
        if not isinstance(request_id, str) or not re.fullmatch(r"[A-Za-z0-9_-]{8,64}", request_id):
            raise BakeryError("INVALID_REQUEST_ID", "Use a request reference of 8-64 letters, digits, underscores or hyphens.")
        pickup = self._pickup(pickup_at)
        canonical = {"product_id": product_id, "quantity": quantity, "customer": customer.model_dump(), "pickup_at": pickup.astimezone(UTC).isoformat()}
        fingerprint = hashlib.sha256(json.dumps(canonical, sort_keys=True).encode()).hexdigest()
        try:
            with closing(self._connect()) as connection, connection:
                connection.execute("BEGIN IMMEDIATE")
                existing = connection.execute("SELECT fingerprint, result_json FROM pickup_inquiries WHERE request_id=?", (request_id,)).fetchone()
                if existing:
                    if existing["fingerprint"] != fingerprint:
                        raise BakeryError("REQUEST_CONFLICT", "This request reference belongs to different details. Use a new reference for a new inquiry.")
                    return PickupInquiry.model_validate_json(existing["result_json"]).model_copy(update={"already_saved": True})
                if pickup.astimezone(UTC) - self.now().astimezone(UTC) < timedelta(hours=24):
                    raise BakeryError("PICKUP_TOO_SOON", "Pickup inquiries require at least 24 hours' notice.")
                if pickup.weekday() == 6 or not time(7) <= pickup.time() < time(19):
                    raise BakeryError("PICKUP_CLOSED", "Choose Monday-Saturday, 07:00 to before 19:00 Beirut time.")
                inquiry = PickupInquiry(inquiry_id="CC-" + uuid4().hex, request_id=request_id,
                    product_id=product.id, product_name=product.name, quantity=quantity,
                    unit_price_usd=product.price_usd, estimated_total_usd=f"{Decimal(product.price_usd) * quantity:.2f}",
                    customer_name=customer.name, pickup_at=pickup.isoformat())
                connection.execute("INSERT INTO pickup_inquiries VALUES (?, ?, ?, ?)", (request_id, fingerprint, customer.email, inquiry.model_dump_json()))
                return inquiry
        except sqlite3.Error as exc:
            raise BakeryError("STORAGE_UNAVAILABLE", "The pickup inquiry could not be saved. Check database access and retry using the same reference.") from exc

    def prepare_pickup_inquiry(self, query: str, quantity: int, customer: Customer, pickup_at: str, request_id: str) -> PreparedInquiry:
        """Coordinate menu lookup followed by the same Action operation."""
        menu = self.search_menu(query)
        if not menu.products:
            raise BakeryError("NO_MATCH", "No menu product matches this request. No inquiry was saved.")
        if len(menu.products) > 1:
            choices = ", ".join(p.name for p in menu.products)
            raise BakeryError("AMBIGUOUS_MATCH", f"Choose one product before saving: {choices}.")
        inquiry = self.save_pickup_inquiry(menu.products[0].id, quantity, customer, pickup_at, request_id)
        return PreparedInquiry(query=menu.query, steps=["search_menu", "save_pickup_inquiry"], inquiry=inquiry)


def next_pickup_at(now: datetime | None = None) -> str:
    """Return a practical future pickup time for the CLI walkthrough."""
    candidate = ((now or datetime.now(UTC)).astimezone(BEIRUT) + timedelta(days=2)).replace(hour=10, minute=0, second=0, microsecond=0)
    while candidate.weekday() == 6:
        candidate += timedelta(days=1)
    return candidate.isoformat()


def inspect_inquiries(database: Path | str) -> list[dict]:
    """Local read-only inspection; not exposed as an MCP tool."""
    uri = Path(database).resolve().as_uri() + "?mode=ro"
    with closing(sqlite3.connect(uri, uri=True)) as connection:
        return [json.loads(row[0]) for row in connection.execute("SELECT result_json FROM pickup_inquiries ORDER BY request_id")]


if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser(description="Inspect saved pickup inquiries.")
    parser.add_argument("database", nargs="?", default=str(ROOT / "data" / "pickup.sqlite3"))
    args = parser.parse_args()
    print(json.dumps(inspect_inquiries(args.database), indent=2))


## 4. MCP Server · server.py
### Configuration and error handling


In [ ]:
%%writefile "{PROJECT}/server.py"
"""Cedar Crumb FastMCP server: one Data, Action, and Orchestration tool."""
import logging
import os
from pathlib import Path
from typing import Callable, TypeVar

from fastmcp import FastMCP
from fastmcp.exceptions import ToolError

from bakery import Bakery, BakeryError
from models import Customer, MenuResult, PickupAt, PickupInquiry, PreparedInquiry, ProductID, Quantity, Query, RequestID

ROOT = Path(__file__).resolve().parent
log = logging.getLogger("cedar_crumb")
T = TypeVar("T")


def create_server(database: Path | str | None = None) -> FastMCP:
    bakery = Bakery(database or os.getenv("BAKERY_DB", str(ROOT / "data" / "pickup.sqlite3")))
    bakery.initialize()
    mcp = FastMCP("Cedar Crumb")

    def invoke(name: str, operation: Callable[[], T]) -> T:
        log.info("SERVER received %s", name)
        try:
            result = operation()
        except BakeryError as exc:
            log.warning("SERVER failed %s: %s", name, exc.code)
            raise ToolError(str(exc)) from exc
        log.info("SERVER completed %s", name)
        return result



### Data · search_menu


In [ ]:
%%writefile -a "{PROJECT}/server.py"
    @mcp.tool(annotations={"readOnlyHint": True, "idempotentHint": True, "openWorldHint": False})
    def search_menu(query: Query = "") -> MenuResult:
        """DATA: Retrieve Cedar Crumb menu names, IDs, USD prices, loaf weights, ingredients, allergens, and pickup policies. Search is a case-insensitive literal substring; an empty query lists all products. No matches returns an empty product list. Whitespace-only or oversized input fails clearly. This lookup does not save an inquiry or check stock."""
        return invoke("search_menu", lambda: bakery.search_menu(query))



### Action · save_pickup_inquiry


In [ ]:
%%writefile -a "{PROJECT}/server.py"
    @mcp.tool(annotations={"readOnlyHint": False, "destructiveHint": False, "idempotentHint": True, "openWorldHint": False})
    def save_pickup_inquiry(product_id: ProductID, quantity: Quantity, customer: Customer, pickup_at: PickupAt, request_id: RequestID) -> PickupInquiry:
        """ACTION: Save one pickup inquiry for a known product ID. Use only after the customer explicitly requests saving and agrees to contact follow-up. Persist the supplied contact details in SQLite; return a reference, menu-price estimate, and pending_confirmation status. Require 24 hours' notice and Monday-Saturday 07:00 to before 19:00 Beirut time. Reusing a reference with identical details returns the existing inquiry; changed details cause REQUEST_CONFLICT. Unknown product, invalid quantity, missing consent, invalid/closed/too-soon pickup, and unavailable storage fail without creating a request. Staff confirmation determines fulfillment."""
        return invoke("save_pickup_inquiry", lambda: bakery.save_pickup_inquiry(product_id, quantity, customer, pickup_at, request_id))



### Orchestration · prepare_pickup_inquiry


In [ ]:
%%writefile -a "{PROJECT}/server.py"
    @mcp.tool(annotations={"readOnlyHint": False, "destructiveHint": False, "idempotentHint": True, "openWorldHint": False})
    def prepare_pickup_inquiry(query: Query, quantity: Quantity, customer: Customer, pickup_at: PickupAt, request_id: RequestID) -> PreparedInquiry:
        """ORCHESTRATION: Prepare a requested pickup inquiry from a product name, flavor, or ingredient query. Call the Data lookup, require exactly one matching product, then call the same Action operation to save it. Return the executed steps and inquiry. NO_MATCH or AMBIGUOUS_MATCH creates no inquiry; clarify the product instead. Consent, pickup validation, storage errors, and retry rules are the same as the Action tool. Requires an explicit request to save."""
        return invoke("prepare_pickup_inquiry", lambda: bakery.prepare_pickup_inquiry(query, quantity, customer, pickup_at, request_id))


### Server startup · stdio


In [ ]:
%%writefile -a "{PROJECT}/server.py"

    return mcp


if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    create_server().run(transport="stdio", show_banner=False)


## 5. Host and MCP Client · client.py

```text
Customer → Host (Agent + Runner) → MCP Client → MCP Server
Customer ← Host (final answer)  ← MCP Client ← MCP Server
```

### MCP Client · subprocess connection


In [ ]:
%%writefile "{PROJECT}/client.py"
"""Cedar Crumb host: OpenAI Agents SDK, Gemini, and a stdio MCP client."""
from __future__ import annotations

import argparse
import ast
import asyncio
import json
import os
import sys
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any

from agents import Agent, Model, ModelSettings, OpenAIChatCompletionsModel, Runner, set_tracing_disabled
from agents.mcp import MCPServerStdio
from dotenv import load_dotenv
from openai import AsyncOpenAI, OpenAIError

from bakery import BEIRUT, inspect_inquiries, next_pickup_at
from uuid import uuid4

ROOT = Path(__file__).resolve().parent
INSTRUCTIONS = """You are Nour, Cedar Crumb's bread guide and pickup assistant.
Be warm, precise, and concise. Use the tools for product names, prices, weights,
ingredients, allergens, policies, and saved inquiry details.
Use search_menu for information only. An empty query lists the full menu.
Use save_pickup_inquiry when the customer explicitly asks to save a known product ID.
Use prepare_pickup_inquiry when asked to find a product by name, flavor, or ingredient
and save its pickup inquiry. This workflow performs lookup and persistence together.
Ask for missing quantity, customer-provided name/email, pickup date/time, or consent.
Save only after explicit agreement to store contact details for pickup follow-up.
Use a new unique request_id of 8-64 allowed characters for a new request; reuse the
same reference when retrying identical details. Honor a supplied request reference.
Use Asia/Beirut for pickup times and pass an explicit ISO 8601 UTC offset.
Require at least 24 hours' notice; pickups run Monday-Saturday, 07:00 to before 19:00.
An inquiry has pending_confirmation status. State that it awaits staff confirmation;
do not claim confirmed stock, fulfillment, payment, email delivery, or notification.
Use the returned USD estimate rather than mental arithmetic. All breads contain gluten;
use the returned ingredient and shared-kitchen information for allergy questions.
If a query is ambiguous, ask the customer to choose; never select a loaf arbitrarily.
Explain errors and request a correction. Never describe a failed save as successful.
After a successful save, report the product, quantity, estimated total, pickup time,
inquiry reference, and pending confirmation. Acknowledge an already_saved retry.
"""


def tool_definition(tool) -> dict:
    return {"name": tool.name, "description": tool.description,
        "input_schema": getattr(tool, "input_schema", None) or getattr(tool, "inputSchema", {}),
        "output_schema": getattr(tool, "output_schema", None) or getattr(tool, "outputSchema", None)}


class ObservableMCPServer(MCPServerStdio):
    """Observe actual transport requests and distinguish cache hits from discovery."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, message_handler=self._notification, **kwargs)
        self.emit = lambda **event: None
        self.connection_id = uuid4().hex[:12]
        self.discovery_requests = 0
        self.tool_calls = 0
        self.turn = 0
        self.discovered_turn = None

    async def _notification(self, message):
        method = getattr(message, "method", None) or getattr(getattr(message, "root", None), "method", None)
        if method == "notifications/tools/list_changed":
            self.invalidate_tools_cache()
            self.emit(direction="MCP Server → MCP Client", operation=method, cache="invalidated")

    async def _list_tools_page(self, session, cursor=None):
        self.discovery_requests += 1
        self.emit(direction="MCP Client → MCP Server", operation="tools/list", transport="stdio", cursor=cursor)
        result = await super()._list_tools_page(session, cursor)
        self.discovered_turn = self.turn
        self.emit(direction="MCP Server → MCP Client", operation="tools/list result",
            definitions=[tool_definition(t) for t in result.tools])
        return result

    async def list_tools(self, *args, **kwargs):
        before = self.discovery_requests
        tools = await super().list_tools(*args, **kwargs)
        if {t.name for t in tools} != {"search_menu", "save_pickup_inquiry", "prepare_pickup_inquiry"}:
            raise RuntimeError("Unexpected MCP tool set; restart after reviewing the server configuration.")
        if self.discovery_requests == before:
            cross_turn = self.discovered_turn is not None and self.discovered_turn < self.turn
            self.emit(direction="MCP Client → Host", operation="cache/reuse",
                scope="cross-turn" if cross_turn else "within-turn",
                cache="Cached tool definitions reused" if cross_turn else "Tool definitions reused within current turn",
                tools=[t.name for t in tools], connection=self.connection_id)
        return tools

    async def call_tool(self, tool_name, arguments, meta=None):
        self.emit(direction="Agent → MCP Client", operation="tool selection", tool=tool_name, arguments=arguments)
        # SDK-side rejections must not be presented as server requests.
        self._validate_required_parameters(tool_name=tool_name, arguments=arguments)
        self.emit(direction="MCP Client → MCP Server", operation="tools/call", tool=tool_name, arguments=arguments)
        self.tool_calls += 1
        result = await super().call_tool(tool_name, arguments, meta)
        payload = getattr(result, "structured_content", None) or getattr(result, "structuredContent", None)
        if payload is None:
            payload = decode_output(next((c.text for c in result.content if c.type == "text"), ""))
        self.emit(direction="MCP Server → MCP Client", operation="tools/call result", result=payload,
            is_error=bool(getattr(result, "is_error", False) or getattr(result, "isError", False)))
        return result


class ObservedModel(Model):
    """Record the exact function schemas supplied to each model request."""
    def __init__(self, delegate, emit):
        self.delegate, self.emit = delegate, emit

    def _request(self, args, kwargs):
        tools = kwargs.get("tools", args[3] if len(args) > 3 else [])
        self.emit(direction="Host → Model", operation="model/request",
            definitions=[{"name": t.name, "description": getattr(t, "description", None),
                "input_schema": getattr(t, "params_json_schema", None)} for t in tools])

    async def get_response(self, *args, **kwargs):
        self._request(args, kwargs)
        return await self.delegate.get_response(*args, **kwargs)

    async def stream_response(self, *args, **kwargs):
        self._request(args, kwargs)
        async for event in self.delegate.stream_response(*args, **kwargs):
            yield event


def connect_server(database: Path | str | None = None) -> ObservableMCPServer:
    env = {"PYTHONIOENCODING": "utf-8", "PYTHONDONTWRITEBYTECODE": "1"}
    if database is not None:
        env["BAKERY_DB"] = str(Path(database).resolve())
    return ObservableMCPServer(name="Cedar Crumb MCP client",
        params={"command": sys.executable, "args": [str(ROOT / "server.py")], "cwd": str(ROOT), "env": env},
        cache_tools_list=True, client_session_timeout_seconds=30)




### Host · agent and model configuration


In [ ]:
%%writefile -a "{PROJECT}/client.py"
def build_agent(server: MCPServerStdio) -> Agent:
    load_dotenv(ROOT / ".env", override=False)
    key = os.getenv("GEMINI_API_KEY", "").strip()
    if not key:
        raise RuntimeError("Set GEMINI_API_KEY in .env. The offline tests and MCP smoke check do not require a key.")
    set_tracing_disabled(True)
    provider = AsyncOpenAI(api_key=key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/", timeout=60, max_retries=1)
    model = OpenAIChatCompletionsModel(model=os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite"), openai_client=provider)
    current = datetime.now(BEIRUT).isoformat()
    return Agent(name="Nour | Cedar Crumb", instructions=INSTRUCTIONS + f"\nCurrent Beirut date/time: {current}",
        model=model, model_settings=ModelSettings(parallel_tool_calls=False), mcp_servers=[server])




### Agent loop and tool results


In [ ]:
%%writefile -a "{PROJECT}/client.py"
def decode_output(raw: Any) -> dict:
    """Normalize the SDK's text-content wrapper for display and local checks."""
    value = raw
    if isinstance(value, str):
        try:
            value = ast.literal_eval(value)
        except (SyntaxError, ValueError):
            pass
    if isinstance(value, dict) and value.get("type") == "text":
        value = value["text"]
    if isinstance(value, str):
        try:
            value = json.loads(value)
        except ValueError:
            return {"error": value.split(":", 1)[0], "message": value}
    return value if isinstance(value, dict) else {"content": value}


@dataclass
class Turn:
    history: list
    calls: list[str]
    outputs: list[dict]
    answer: str


async def run_turn(agent: Agent, history: list, prompt: str) -> Turn:
    print(f"\nCUSTOMER > {prompt}", flush=True)
    result = Runner.run_streamed(agent, history + [{"role": "user", "content": prompt}], max_turns=8)
    calls, outputs = [], []
    async for event in result.stream_events():
        if event.type != "run_item_stream_event":
            continue
        if event.name == "tool_called":
            raw = event.item.raw_item
            calls.append(getattr(raw, "name", "unknown"))
            print(f"AGENT -> MCP CLIENT -> SERVER: {calls[-1]} {getattr(raw, 'arguments', '')}", flush=True)
        elif event.name == "tool_output":
            data = decode_output(event.item.output)
            outputs.append(data)
            print("SERVER -> MCP CLIENT -> AGENT:", json.dumps(data, ensure_ascii=False), flush=True)
    print(f"NOUR > {result.final_output}", flush=True)
    return Turn(result.to_input_list(), calls, outputs, str(result.final_output))


def walkthrough_steps(pickup_at: str) -> list[tuple[str, str, str | None, int]]:
    customer = f"My name is Alex, email alex@example.com. Pickup: {pickup_at}. I agree to save my contact details for pickup follow-up."
    return [
        ("Which sourdough loaves contain sesame? Only show menu information; do not save a request.", "search_menu", None, 0),
        (f"Save a pickup inquiry for 2 loaves of product ID plain. {customer} Request reference cedar-action-01.", "save_pickup_inquiry", None, 1),
        (f"Find the Olive & Rosemary loaf and save a pickup inquiry for 3 loaves. {customer} Request reference cedar-workflow-01.", "prepare_pickup_inquiry", None, 2),
        (f"Find a chocolate croissant and save a pickup inquiry for 1. {customer} Request reference cedar-missing-01.", "prepare_pickup_inquiry", "NO_MATCH", 2),
        (f"Save an inquiry for 2 loaves of product ID plain again. This retries exactly the same request. {customer} Reuse request reference cedar-action-01.", "save_pickup_inquiry", None, 2),
    ]


async def main(args: argparse.Namespace) -> None:
    load_dotenv(ROOT / ".env", override=False)
    database = (args.database or ROOT / "data" / "pickup.sqlite3").resolve()
    if args.demo and database.exists() and inspect_inquiries(database):
        raise RuntimeError("Choose a fresh --database filename for the walkthrough so its row-count checks are reproducible.")
    async with connect_server(database) as server:
        tools = await server.list_tools()
        names = {tool.name for tool in tools}
        if names != {"search_menu", "save_pickup_inquiry", "prepare_pickup_inquiry"}:
            raise RuntimeError(f"Unexpected MCP tool set: {names}")
        print("MCP tools/list:", ", ".join(sorted(names)), flush=True)
        agent = build_agent(server)
        history: list = []
        if args.demo:
            first_id = None
            for index, (prompt, expected, error, expected_rows) in enumerate(walkthrough_steps(next_pickup_at())):
                turn = await run_turn(agent, history, prompt)
                history = turn.history
                if turn.calls != [expected] or len(turn.outputs) != 1:
                    raise RuntimeError(f"Step {index + 1}: expected {expected}, got {turn.calls}.")
                output = turn.outputs[0]
                if error and output.get("error") != error:
                    raise RuntimeError(f"Step {index + 1}: expected {error}, got {output}.")
                if not error and "error" in output:
                    raise RuntimeError(f"Step {index + 1} failed: {output}.")
                rows = inspect_inquiries(database)
                if len(rows) != expected_rows:
                    raise RuntimeError(f"Step {index + 1}: expected {expected_rows} saved inquiries, got {len(rows)}.")
                if index == 1:
                    first_id = output["inquiry_id"]
                    assert output["estimated_total_usd"] == "12.00"
                if index == 2:
                    assert output["steps"] == ["search_menu", "save_pickup_inquiry"]
                    assert output["inquiry"]["estimated_total_usd"] == "24.00"
                if index == 4:
                    assert output["inquiry_id"] == first_id and output["already_saved"]
                print(f"CHECK: correct tool; {len(rows)} saved inquiries.", flush=True)
            print("\nWalkthrough passed: three tools, a clear failure, persistent requests, and a safe retry.", flush=True)
        elif args.prompt:
            await run_turn(agent, history, args.prompt)
        else:
            print("Cedar Crumb | Good bread. Time well spent.\nType quit to exit.", flush=True)
            while True:
                try:
                    prompt = await asyncio.to_thread(input, "\nYou: ")
                except EOFError:
                    break
                if prompt.strip().casefold() in {"quit", "exit"}:
                    break
                if prompt.strip():
                    history = (await run_turn(agent, history, prompt)).history


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    mode = parser.add_mutually_exclusive_group()
    mode.add_argument("--demo", action="store_true", help="Run a checked live walkthrough with a fresh database.")
    mode.add_argument("--prompt")
    parser.add_argument("--database", type=Path)
    try:
        asyncio.run(main(parser.parse_args()))
    except (RuntimeError, OSError, OpenAIError, AssertionError) as exc:
        print(f"ERROR: {exc}", file=sys.stderr)
        sys.exit(1)


## 6. Session connection and schema cache · session_runtime.py


In [ ]:
%%writefile "{PROJECT}/session_runtime.py"
"""One owner task manages each chat's MCP connection and schema cache."""
from __future__ import annotations

import asyncio
from dataclasses import dataclass

from agents import Runner

from client import ObservedModel, connect_server, decode_output


@dataclass
class Request:
    message: str
    history: list
    agent_factory: object
    updates: asyncio.Queue


class SessionRuntime:
    def __init__(self, database, *, idle_timeout=900):
        self.database = database
        self.loop = asyncio.get_running_loop()
        self.jobs = asyncio.Queue()
        self.idle_timeout = idle_timeout
        self.server = None
        self.closing = False
        self.close_error = None
        self.owner = asyncio.create_task(self._serve())

    async def _serve(self):
        """Connection entry, requests, and cleanup all execute in this task."""
        current = None
        try:
            # Delay connection startup until the first real request.
            current = await asyncio.wait_for(self.jobs.get(), self.idle_timeout)
            if current is None:
                return
            async with connect_server(self.database) as server:
                self.server = server
                while current is not None:
                    server.turn += 1
                    server.emit = lambda **event: current.updates.put_nowait(("event", event))
                    try:
                        await server.list_tools()
                        agent = current.agent_factory(server)
                        agent.model = ObservedModel(agent.model, server.emit)
                        result = Runner.run_streamed(agent, current.history + [{"role": "user", "content": current.message}], max_turns=8)
                        async for event in result.stream_events():
                            if event.type == "run_item_stream_event" and event.name == "tool_output":
                                server.emit(direction="MCP Client → Agent", operation="tool result delivered",
                                    result=decode_output(event.item.output))
                        current.updates.put_nowait(("answer", (str(result.final_output), result.to_input_list())))
                    except Exception as error:
                        # Retire the connection after a failed turn; never claim a stale cache hit.
                        self.closing = True
                        current.updates.put_nowait(("error", error))
                        return
                    finally:
                        server.emit = lambda **event: None
                        current.updates.put_nowait(("done", None))
                        current = None
                    current = await asyncio.wait_for(self.jobs.get(), self.idle_timeout)
        except TimeoutError:
            pass
        except BaseException as error:
            self.close_error = error
            if current is not None:
                current.updates.put_nowait(("error", error))
                current.updates.put_nowait(("done", None))
        finally:
            self.closing = True
            self.server = None
            while not self.jobs.empty():
                pending = self.jobs.get_nowait()
                if pending is not None:
                    pending.updates.put_nowait(("error", RuntimeError("MCP connection closed; retry to reconnect.")))
                    pending.updates.put_nowait(("done", None))

    async def run(self, message, history, agent_factory):
        if self.closing or self.owner.done() or self.loop is not asyncio.get_running_loop():
            raise RuntimeError("MCP session is closed or belongs to a different event loop.")
        updates = asyncio.Queue()
        self.jobs.put_nowait(Request(message, list(history), agent_factory, updates))
        while True:
            kind, payload = await updates.get()
            if kind == "done":
                break
            yield kind, payload

    def request_close(self):
        if not self.closing and not self.loop.is_closed():
            self.closing = True
            self.loop.call_soon_threadsafe(self.jobs.put_nowait, None)

    async def close(self):
        self.request_close()
        if self.loop is asyncio.get_running_loop():
            await self.owner


### Chat and request activity · app.py


In [ ]:
%%writefile "{PROJECT}/app.py"
"""Interactive Cedar Crumb host with actual MCP activity and saved inquiries."""
from __future__ import annotations

import argparse
import asyncio
import json
import os
from pathlib import Path
from uuid import uuid4

os.environ.setdefault("GRADIO_ANALYTICS_ENABLED", "False")
import gradio as gr
from openai import AuthenticationError, RateLimitError, NotFoundError

from bakery import ROOT, inspect_inquiries, next_pickup_at
from client import build_agent, walkthrough_steps
from session_runtime import SessionRuntime

RUNTIMES = {}

CATEGORIES = {"search_menu": "Data", "save_pickup_inquiry": "Action", "prepare_pickup_inquiry": "Orchestration"}


def new_session():
    return {"history": [], "database": str(ROOT / "results" / f"chat-{uuid4().hex}.sqlite3"), "events": []}


def records(database):
    if not Path(database).exists():
        return []
    return [[r["inquiry_id"], r["product_name"], r["quantity"], r["estimated_total_usd"], r["pickup_at"], r["status"]]
            for r in inspect_inquiries(database)]


def activity_rows(events):
    if not events:
        return []
    current = [e for e in events if e["request"] == events[-1]["request"]]
    rows = []
    for index, event in enumerate(current, 1):
        detail = event.get("cache") or event.get("tool") or ""
        if "definitions" in event:
            detail = ", ".join(t["name"] for t in event["definitions"])
        if "result" in event:
            detail = "Error returned" if event.get("is_error") else "Result returned"
        rows.append([str(index), event["direction"], event.get("operation", "message"), detail])
    return rows


def runtime_for(session):
    key = session["database"]
    runtime = RUNTIMES.get(key)
    loop = asyncio.get_running_loop()
    if runtime is None or runtime.closing or runtime.owner.done() or runtime.loop is not loop:
        if runtime is not None:
            runtime.request_close()
        runtime = SessionRuntime(key)
        RUNTIMES[key] = runtime
    return runtime


def release_session(session):
    if session:
        runtime = RUNTIMES.pop(session["database"], None)
        if runtime is not None:
            runtime.request_close()


async def reset_session(session):
    runtime = RUNTIMES.pop(session["database"], None) if session else None
    if runtime is not None:
        await runtime.close()
    return "", [], new_session(), [], []


async def clear_session(session):
    _, _, fresh, events, rows = await reset_session(session)
    return fresh, events, rows


async def respond(message, chat, session):
    session = session or new_session()
    chat = list(chat or [])
    if not message.strip():
        yield "", chat, session, session["events"], records(session["database"])
        return
    request_ref = "turn-" + uuid4().hex[:8]
    def record(direction, **details):
        if "tool" in details:
            details["category"] = CATEGORIES.get(details["tool"])
        session["events"].append({"request": request_ref, "direction": direction, **details})
    record("Customer → Host", operation="customer request", message=message)
    chat += [{"role": "user", "content": message}, {"role": "assistant", "content": "Checking your request…"}]
    yield "", chat, session, session["events"], records(session["database"])
    try:
        runtime = runtime_for(session)
        async for kind, payload in runtime.run(message, session["history"], build_agent):
            if kind == "event":
                record(**payload)
                yield "", chat, session, session["events"], records(session["database"])
            elif kind == "answer":
                answer, session["history"] = payload
                chat[-1] = {"role": "assistant", "content": answer}
                record("Host → Customer", operation="final answer", answer=answer)
            elif kind == "error":
                raise payload
    except Exception as error:
        if isinstance(error, AuthenticationError):
            explanation = "Gemini rejected the API key. Check GEMINI_API_KEY and restart the app."
        elif isinstance(error, RateLimitError):
            explanation = "Gemini's rate or quota limit was reached. Check your API quota and retry later."
        elif isinstance(error, NotFoundError):
            explanation = "The configured Gemini model is unavailable. Check GEMINI_MODEL and restart the app."
        else:
            explanation = "The request could not finish. Check the runtime logs and connection. Inspect saved inquiries before retrying; reuse the same request reference."
        chat[-1] = {"role": "assistant", "content": explanation}
        record("Host → Customer", operation="error", error=explanation)
    yield "", chat, session, session["events"], records(session["database"])


def create_app():
    menu = json.loads((ROOT / "data/bakery.json").read_text(encoding="utf-8"))
    with gr.Blocks(title="Cedar Crumb") as app:
        gr.Markdown("# Cedar Crumb\n**Good bread. Time well spent.**\n\nExplore our breads and prepare your pickup inquiry with Nour.")
        session = gr.State(value=new_session, time_to_live=900, delete_callback=release_session)
        with gr.Row():
            with gr.Column(scale=3):
                chat = gr.Chatbot(label="Nour · Bread guide & pickup assistant", height=450)
                message = gr.Textbox(label="Your message", placeholder="Ask about our breads, or arrange a pickup inquiry…", lines=2)
                with gr.Row():
                    send = gr.Button("Send", variant="primary")
                    reset = gr.Button("New conversation")
                gr.Examples(examples=[[p[0]] for p in walkthrough_steps(next_pickup_at())], inputs=message, label="Try a request · click, then Send")
            with gr.Column(scale=2):
                with gr.Tab("Menu"):
                    gr.Dataframe(value=[[p["name"], p["weight_g"], p["price_usd"]] for p in menu["products"]],
                        headers=["Bread", "Weight (g)", "USD"], interactive=False)
                    gr.Markdown("**Hamra, Beirut** · Monday–Saturday, 07:00–19:00.\n\nPickup inquiries need 24 hours' notice and await staff confirmation. All breads contain gluten; ask Nour for ingredients and allergen details.")
                with gr.Tab("Tool activity"):
                    timeline = gr.Dataframe(value=[], headers=["Step", "Direction", "Operation", "Detail"],
                        label="Current request", interactive=False)
                    activity = gr.JSON(value=[], label="Actual MCP calls and returned results")
                with gr.Tab("Saved inquiries"):
                    saved = gr.Dataframe(headers=["Reference", "Bread", "Quantity", "USD estimate", "Pickup", "Status"], value=[], interactive=False)
                    refresh = gr.Button("Refresh inquiries")
        gr.Markdown("Contact details are saved only with your agreement for pickup follow-up. Menu information and conversation content are processed by Gemini.")
        inputs = [message, chat, session]
        outputs = [message, chat, session, activity, saved]
        send.click(respond, inputs, outputs, concurrency_limit=1, concurrency_id="chat")
        message.submit(respond, inputs, outputs, concurrency_limit=1, concurrency_id="chat")
        reset.click(reset_session, inputs=session, outputs=outputs,
            concurrency_limit=1, concurrency_id="chat")
        chat.clear(clear_session, inputs=session, outputs=[session, activity, saved],
            concurrency_limit=1, concurrency_id="chat")
        refresh.click(lambda s: records(s["database"]), session, saved)
        activity.change(activity_rows, inputs=activity, outputs=timeline, queue=False)
    return app


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--share", action="store_true")
    parser.add_argument("--port", type=int, default=7860)
    args = parser.parse_args()
    create_app().queue().launch(server_name="127.0.0.1", server_port=args.port, share=args.share,
        theme=gr.themes.Soft(primary_hue="amber", neutral_hue="stone"), show_error=False)


## 7. Tool discovery


In [ ]:
discovery_script = """import asyncio
import json
from client import connect_server, tool_definition

async def discover():
    async with connect_server('results/discovery.sqlite3') as server:
        tools = await server.list_tools()
        assert len(tools) == 3
        print(json.dumps([tool_definition(t) for t in tools], indent=2))
        print('tools/list: exactly three registered tools; stdio connection completed.')

asyncio.run(discover())
"""
command([PYTHON, '-c', discovery_script])


## 8. Tests


In [ ]:
command([PYTHON, '-m', 'pytest', '-q'])
command([PYTHON, 'smoke_test.py'])


## 9. Interactive application


In [ ]:
import re, time
from IPython.display import HTML, display
if 'APP_PROCESS' in globals() and APP_PROCESS.poll() is None:
    APP_PROCESS.terminate()
    APP_PROCESS.wait(timeout=20)
LOG_PATH = PROJECT / 'app.log'
with LOG_PATH.open('w') as log:
    APP_PROCESS = subprocess.Popen([PYTHON, '-u', 'app.py', '--share'], cwd=PROJECT, stdout=log, stderr=subprocess.STDOUT)
url = None
for _ in range(180):
    text = LOG_PATH.read_text(errors='replace')
    match = re.search(r'https://[a-zA-Z0-9-]+\.gradio\.live', text)
    if match:
        url = match.group(0)
        break
    if APP_PROCESS.poll() is not None:
        raise RuntimeError('App startup failed: ' + text)
    time.sleep(1)
if not url:
    raise RuntimeError('No share link received: ' + LOG_PATH.read_text())
print('Open Cedar Crumb:', url)
display(HTML(f'<p><a href="{url}" target="_blank">Open Cedar Crumb in a new tab</a></p><iframe src="{url}" width="100%" height="1000" style="border:0"></iframe>'))
